# PriorModel — Colab (TE+TM, dx=160 m, commemi-every 5)

U-Net eğitimi Colab GPU'da; COMMEMI / VFSA tam değerlendirmesi **yerelde** (`scripts/evaluate_mid_scale_v8.jl`). Bu defter **yalnızca** planlanan v8 koşusunu çalıştırır.

**Bu koşu (başka flag yok):** n=200, `--tetm`, `UNET_MESH` = v4–v7 (`nx=120`, `dx=160`), 15 epoch, `--commemi-every 5`.

Atılan kalıntılar — **kullanmayın:** `train_pairs_v8_tetm_n200.h5`, `prior_v8_tetm_n200.jld2`, `--commemi-every 0`, `--no-plot`, `--epochs 50`, dx=80 / nx=240.

1. `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 yeterli).
2. Kernel **Python 3** kalsın.
3. `cd /content/PriorModel` sonra tam olarak `julia --project=. …` (cwd `/content` iken `--project=.` yasak).


## Önce `cd /content/PriorModel`, sonra `--project=.`

`/content` içinde `julia --project=.` boş bir `/content/Project.toml` bırakır. Klon dizinine geçin:

```bash
cd /content/PriorModel
julia --project=. …
```


In [ ]:
%%bash
set -euo pipefail
if [[ -f /content/Project.toml ]]; then
  echo "Removing leftover /content/Project.toml (created by --project=. from /content)"
  rm -f /content/Project.toml /content/Manifest.toml
fi
ls -la /content/Project.toml 2>/dev/null || echo "No /content/Project.toml — good."

## Julia 1.12.4 kur

Colab'ın yerleşik Julia runtime'ı 1.10 LTS; bu proje Julia 1.11+ ister (`Project.toml`). 1.12.4 Linux x86_64 tarball'ını `/usr/local`'a açıyoruz.

In [ ]:
%%bash
set -euo pipefail
JULIA_VERSION="1.12.4"
JULIA_VER="${JULIA_VERSION%.*}"
if julia --version 2>/dev/null | grep -q "${JULIA_VERSION}"; then
  julia --version
  exit 0
fi
echo "Installing Julia ${JULIA_VERSION}…"
URL="https://julialang-s3.julialang.org/bin/linux/x64/${JULIA_VER}/julia-${JULIA_VERSION}-linux-x86_64.tar.gz"
wget -q "${URL}" -O /tmp/julia.tar.gz
tar -xzf /tmp/julia.tar.gz -C /usr/local --strip-components=1
rm /tmp/julia.tar.gz
julia --version

## Repoyu klonla

Public HTTPS clone kullanıcı adı sormasın diye `GIT_TERMINAL_PROMPT=0`. 90 saniye takılırsa `master.zip` indirilir.

In [ ]:
%%bash
set -euo pipefail
export GIT_TERMINAL_PROMPT=0
export GIT_PAGER=cat
unset GIT_ASKPASS SSH_ASKPASS

REPO_URL="https://github.com/hayrunnisayildiz/PriorModel.git"
ZIP_URL="https://github.com/hayrunnisayildiz/PriorModel/archive/refs/heads/master.zip"
DEST="/content/PriorModel"

echo "=== public clone (no login) ==="

if [[ -d "${DEST}/.git" ]]; then
  echo "Updating existing clone…"
  git -C "${DEST}" -c credential.helper= --no-pager fetch --depth 1 origin master
  git -C "${DEST}" --no-pager reset --hard FETCH_HEAD
else
  rm -rf "${DEST}"
  echo "Cloning ${REPO_URL} …"
  if timeout 90 git -c credential.helper= clone --depth 1 --single-branch --branch master --progress "${REPO_URL}" "${DEST}"; then
    echo "git clone OK"
  else
    echo "git clone stalled/failed — downloading public zip instead"
    rm -rf "${DEST}"
    wget -q --show-progress -O /tmp/PriorModel.zip "${ZIP_URL}"
    unzip -qo /tmp/PriorModel.zip -d /tmp
    mv /tmp/PriorModel-master "${DEST}"
    rm -f /tmp/PriorModel.zip
  fi
fi

test -f "${DEST}/Project.toml"
ls -l "${DEST}/Project.toml"
echo "OK"

## Mesh doğrula (v4–v7 UNET_MESH)

`UNET_MESH` **nx=120, dx=160 m** olmalı. dx=80 / nx=240 yaması bu koşuda yok.

In [ ]:
from pathlib import Path

p = Path("/content/PriorModel/src/synthetic/MeshParams.jl")
text = p.read_text()
start = text.find("const UNET_MESH")
assert start >= 0, "UNET_MESH not found in MeshParams.jl"
block = text[start:start + 500]
print(block)
assert "120," in block and "160.0," in block, block
assert "240," not in block, "UNET_MESH still has nx=240"
assert "80.0," not in block, "UNET_MESH still has dx=80 m"
print("OK: UNET_MESH is 120 × 160 m (v4–v7 contract)")

## Google Drive'ı bağla (sadece sonuç kopyası)

Eski `train_pairs_v8_tetm_n200.h5` / `prior_v8_tetm_n200.jld2` **kopyalanmaz, eğitime alınmaz**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Julia projesini instantiate et

`MTGeophysics.jl` v0.4.2 GitHub tag'inden gelir. Burada `using MTGeophysics` yapmayın.

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul
echo "=== Pkg.instantiate (several minutes; no output is normal) ==="
cd /content/PriorModel
julia --project=. -e '
println("julia started; instantiating…"); flush(stdout); flush(stderr)
include("src/pkg_setup.jl")
println("active=", Base.active_project())
flush(stdout)
'

## GPU kontrolü

`Device: CPU` veya `LuxCUDA unavailable` görürseniz Runtime tipini GPU yapıp kernel'ı restart edin.

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul

echo "=== Colab runtime ==="
nvidia-smi -L || { echo "nvidia-smi missing: Runtime is not GPU. Runtime → Change runtime type → GPU, then restart."; exit 1; }
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

echo
echo "=== Julia CUDA ==="
cd /content/PriorModel
julia --project=. -e '
println("julia started"); flush(stdout)
try
    using LuxCUDA
catch err
    println("LuxCUDA failed: ", err)
    println("→ training will be CPU")
    exit(1)
end
println("CUDA.functional() = ", CUDA.functional())
if CUDA.functional()
    println("GPU = ", CUDA.name(CUDA.device()))
    println("OK: training will use CUDA GPU")
else
    println("CUDA.jl loaded but no usable GPU (CUDA.functional()=false)")
    try
        CUDA.versioninfo()
    catch e
        println(e)
    end
    exit(1)
end
'

## HDF5 üret (`--tetm`, n=200, dx=160)

Komut birebir (cwd = repo kökü). `xvfb-run` yalnızca GLMakie display sarmalayıcısıdır; Julia argümanlarına dokunulmaz. Logda `UNET_MESH: nx=120  dx=160.0 m` görünmeli.

In [ ]:
%%bash
set -euo pipefail
export GKSwstype=nul
export JULIA_PKG_PRECOMPILE_AUTO=0
cd /content/PriorModel

H5=data/synthetic/train_pairs_v8_tetm_n200_dx160.h5
if [[ -f "$H5" ]]; then
  echo "already have $H5 ($(du -h "$H5" | cut -f1)) — skip build"
  exit 0
fi

echo "=== install xvfb (GLMakie needs a fake display) ==="
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xvfb >/dev/null

mkdir -p data/synthetic
echo "=== exact command ==="
echo "julia --project=. scripts/build_train_pairs.jl --n 200 --seed 42 --tetm --out data/synthetic/train_pairs_v8_tetm_n200_dx160.h5"
echo "=== build n=200 --tetm dx=160 (v4–v7 UNET_MESH) ==="
script -q -c 'xvfb-run -a julia --project=. \
  scripts/build_train_pairs.jl \
  --n 200 --seed 42 --tetm \
  --out data/synthetic/train_pairs_v8_tetm_n200_dx160.h5' /tmp/build_v8_dx160.log

echo "=== mesh lines from production log ==="
grep -E "UNET_MESH|dx=|nx=" /tmp/build_v8_dx160.log | head -20
grep -q "dx=160.0 m" /tmp/build_v8_dx160.log
grep -q "nx=120" /tmp/build_v8_dx160.log
ls -lh "$H5"
echo "OK: build log has UNET_MESH nx=120 dx=160.0 m"

## Eğit (15 epoch, --commemi-every 5)

`--commemi-every 0` ve `--no-plot` **yok**. Probe Faz 1 yolunu kullanır (val_loss-only fallback değil).

İlk epoch CUDA/Zygote derler; 10–20 dk çıktısız donma normal. Sırayla bakın: `Device: CUDA GPU` → `commemi_every=5` → `epoch 1/15 starting`.

In [ ]:
%%bash
set -euo pipefail
export GKSwstype=nul
export JULIA_PKG_PRECOMPILE_AUTO=1
cd /content/PriorModel

H5=data/synthetic/train_pairs_v8_tetm_n200_dx160.h5
test -f "$H5" || { echo "missing $H5 — run the xvfb build cell first"; exit 1; }
mkdir -p models results

echo "=== exact command ==="
echo "julia --project=. src/training/train_mt_resistivity.jl --dataset data/synthetic/train_pairs_v8_tetm_n200_dx160.h5 --epochs 15 --commemi-every 5 --output models/prior_v8_tetm_n200_dx160.jld2 --training-log results/training_log_v8_tetm_n200_dx160.csv --split-json results/train_val_split_v8_tetm_n200_dx160.json --curve-png results/training_curve_v8_tetm_n200_dx160.png"
echo "=== train TE+TM  15 ep  commemi-every 5 ==="
script -q -c "julia --project=. \
  src/training/train_mt_resistivity.jl \
  --dataset ${H5} \
  --epochs 15 --commemi-every 5 \
  --output models/prior_v8_tetm_n200_dx160.jld2 \
  --training-log results/training_log_v8_tetm_n200_dx160.csv \
  --split-json results/train_val_split_v8_tetm_n200_dx160.json \
  --curve-png results/training_curve_v8_tetm_n200_dx160.png" /tmp/train_v8_dx160.log

echo "=== command/log check ==="
grep -E "commemi_every=5|--commemi-every 5|epochs=15|dx=160" /tmp/train_v8_dx160.log | head -20
grep -q "commemi_every=5" /tmp/train_v8_dx160.log
grep -q "dx=160" /tmp/train_v8_dx160.log
if grep -q "commemi-every 0" /tmp/train_v8_dx160.log; then echo "FAIL: commemi-every 0 leaked in"; exit 1; fi
echo "OK: train log has commemi_every=5 and dx=160"

## Checkpoint dört değeri (evaluate'den ÖNCE)

`cfg.epochs`, `cfg.commemi_every`, `best_commemi_rms`, `mesh_params.dx` planla eşleşmeli: 15 / 5 / sonlu RMS / 160.

In [ ]:
%%bash
set -euo pipefail
export GKSwstype=nul
cd /content/PriorModel
julia --project=. -e '
using JLD2
ckpt = JLD2.load("models/prior_v8_tetm_n200_dx160.jld2")
cfg = ckpt["cfg"]
mp  = ckpt["mesh_params"]
rms = ckpt["best_commemi_rms"]
println("cfg.epochs        = ", cfg.epochs)
println("cfg.commemi_every = ", cfg.commemi_every)
println("best_commemi_rms  = ", rms)
println("mesh dx           = ", mp.dx)
println("mesh nx           = ", mp.nx)
cfg.epochs == 15 || error("epochs=$(cfg.epochs) ≠ 15")
cfg.commemi_every == 5 || error("commemi_every=$(cfg.commemi_every) ≠ 5")
mp.dx == 160.0 || error("dx=$(mp.dx) ≠ 160")
isfinite(Float64(rms)) || error("best_commemi_rms is not finite (probe did not run)")
println("OK: checkpoint matches plan (15 / 5 / finite RMS / dx=160)")
'

## Checkpoint + HDF5'i Drive'a kopyala

Yerelde COMMEMI eval (dört değer OK olduktan sonra):

```bash
julia --project=. scripts/evaluate_mid_scale_v8.jl \
    models/prior_v8_tetm_n200_dx160.jld2 results/evaluate_v8_tetm_dx160
```


In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/content/PriorModel")
CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/PriorModelData/PriorModel/PriorModel"),
    Path("/content/drive/MyDrive/PriorModelData/PriorModel"),
    Path("/content/drive/MyDrive/PriorModel"),
]
DRIVE = next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
DRIVE.mkdir(parents=True, exist_ok=True)

rels = (
    "models/prior_v8_tetm_n200_dx160.jld2",
    "data/synthetic/train_pairs_v8_tetm_n200_dx160.h5",
    "results/training_log_v8_tetm_n200_dx160.csv",
    "results/train_val_split_v8_tetm_n200_dx160.json",
    "results/training_curve_v8_tetm_n200_dx160.png",
)

print(f"=== copy dx=160 artifacts → {DRIVE} ===", flush=True)
for rel in rels:
    src = ROOT / rel
    dst = DRIVE / rel
    if not src.is_file():
        print(f"skip (missing): {src}", flush=True)
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"OK {dst} ({dst.stat().st_size / 1e6:.2f} MB)", flush=True)